# 🔍 Scout Agent Test - Job Discovery Engine

**Market Mapper: Real-time Job Discovery using Perplexity Search**

This notebook tests the Scout agent that:
- Uses **Perplexity's real-time web search** to find current job listings
- Matches opportunities to candidate's skills from resume analysis
- Performs multi-source searches (target company, competitors, skill-based)
- Outputs structured discovery results with direct links


## 📦 Step 0: Install Dependencies

Install requests and python-dotenv for OpenRouter API access.

**Important**: After running the installation cell below, if you get a `ModuleNotFoundError`, restart the Jupyter kernel (Kernel → Restart) and run the cells again.


In [1]:
# Install OpenRouter dependencies
%pip install requests python-dotenv --upgrade -q

# Verify installation
try:
    import requests
    from dotenv import load_dotenv
    print('✅ requests and python-dotenv installed successfully')
except ImportError:
    print('❌ Installation failed. Please restart kernel and try again.')
    raise


Note: you may need to restart the kernel to use updated packages.
✅ requests and python-dotenv installed successfully



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 🔑 Step 1: Configure OpenRouter API Key

Make sure your `.env` file contains: `OPENROUTER_API_KEY=your_key_here`


In [8]:
import os
from dotenv import load_dotenv
import requests

load_dotenv()

api_key = os.getenv('OPENROUTER_API_KEY')
if not api_key:
    raise ValueError(
        "OPENROUTER_API_KEY not found in environment variables.\n"
        "Please add it to your .env file: OPENROUTER_API_KEY=sk-or-v1-xxxxx"
    )

# Validate API key format
if not api_key.startswith('sk-or-'):
    print("⚠️  WARNING: API key doesn't start with 'sk-or-'")
    print("   Make sure you're using an OpenRouter API key from https://openrouter.ai/keys")
    print("   NOT from openrouter.io")

print(f"✅ API Key loaded (length: {len(api_key)} chars)")
print(f"   Key prefix: {api_key[:10]}...")

# OpenRouter API client
class OpenRouterClient:
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = "https://openrouter.ai/api/v1"
    
    def create_message(self, model: str, messages: list, **kwargs):
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
            "HTTP-Referer": "https://github.com",
            "X-Title": "Scout Agent"
        }
        payload = {
            "model": model,
            "messages": messages,
            **kwargs
        }
        
        url = f"{self.base_url}/chat/completions"
        response = requests.post(url, headers=headers, json=payload, timeout=120)
        
        if response.status_code == 401:
            error_msg = response.json() if response.text else {}
            print(f"❌ Authentication Failed!")
            print(f"   Error: {error_msg}")
            print(f"\n🔧 Fix:")
            print(f"   1. Get a valid key from: https://openrouter.ai/keys")
            print(f"   2. Make sure you have credits: https://openrouter.ai/credits")
            print(f"   3. Update your .env file with the new key")
            print(f"   4. Restart the kernel and try again")
            response.raise_for_status()
        elif response.status_code != 200:
            print(f"DEBUG: Status Code: {response.status_code}")
            print(f"DEBUG: Response: {response.text[:500]}")
            response.raise_for_status()
        
        return response.json()

client = OpenRouterClient(api_key)
print("✅ OpenRouter client initialized.")
print("\n🧪 Testing connection...")

# Quick test
try:
    test_response = requests.get(
        "https://openrouter.ai/api/v1/models",
        headers={"Authorization": f"Bearer {api_key}"},
        timeout=10
    )
    if test_response.status_code == 200:
        print("✅ API key is valid and working!")
    else:
        print(f"⚠️  API test returned status: {test_response.status_code}")
        print(f"   Response: {test_response.text[:200]}")
except Exception as e:
    print(f"⚠️  Could not test API connection: {e}")


✅ API Key loaded (length: 73 chars)
   Key prefix: sk-or-v1-d...
✅ OpenRouter client initialized.

🧪 Testing connection...
✅ API key is valid and working!


## ⚠️ API Key Troubleshooting

If you get a 401 error, your API key is invalid. Follow these steps:

1. **Get a Valid OpenRouter API Key:**
   - Visit: https://openrouter.ai/keys
   - Sign up/Login (it's free to start)
   - Create a new API key
   - Add credits (minimum $5) at: https://openrouter.ai/credits

2. **Update Your .env File:**
   ```
   OPENROUTER_API_KEY=sk-or-v1-xxxxxxxxxxxxxxxxxxxxx
   ```

3. **Restart the Kernel** and re-run from Step 0

**Note:** The key must start with `sk-or-v1-` and be from openrouter.ai, not openrouter.io


## 🤖 Step 2: Create the Scout Agent

The Scout agent uses **Perplexity via OpenRouter** which has built-in real-time web search capabilities.


In [9]:
# Use Perplexity model via OpenRouter - has built-in web search
SCOUT_MODEL = "perplexity/llama-3.1-sonar-large-128k-online"

SCOUT_SYSTEM_PROMPT = """You are the "Market Mapper" Scout, a high-speed recruitment intelligence agent. 
Your purpose is to bridge the gap between a candidate's current profile and the real-time 2026 job market.

**You have access to real-time web search. Use it to find current job listings.**

Operational Protocol:
1. **JSON Data**: The candidate's resume JSON will be provided in the user message. Analyze it directly from the text.

2. **Web Search**: Use your web search capability to:
   - Search for jobs at the target company
   - Find similar roles at competitors
   - Search for roles matching candidate skills

3. **Match Logic**: 
   - High: >80% overlap with candidate's skills
   - Medium: 50-80% overlap
   - Strategic Pivot: Uses candidate's unique niche skills

4. **Output**: Provide results in a Markdown table with:
   - Job Title
   - Company
   - Match Level (High/Medium/Strategic Pivot)
   - Why it Matches (reference specific skills)
   - Source Link (actual URL from search)

5. **Constraints**: Stay objective. Focus on discovery. Provide actual verified URLs."""

print(f'✅ Scout Agent configured')
print(f'   Model: {SCOUT_MODEL}')
print(f'   Provider: OpenRouter (Perplexity)')
print(f'   Feature: Built-in web search')


✅ Scout Agent configured
   Model: perplexity/llama-3.1-sonar-large-128k-online
   Provider: OpenRouter (Perplexity)
   Feature: Built-in web search


## 📋 Step 3: Prepare Input Data

Provide the Auditor's analysis JSON (resume and JD analysis) and target company.

In [10]:
import json
from pathlib import Path

# Example Job Description (fallback if no JD is loaded from file)
auditor_jd_json = {
    "job_title": "Senior Data Scientist",
    "company": "Google",
    "required_skills": ["Python", "Machine Learning", "Data Analysis", "SQL"],
    "preferred_skills": ["Apache Spark", "AWS", "Team Management"],
    "description": "Looking for senior data scientist with experience in machine learning and data pipelines"
}

# Initialize resume variable
auditor_resume_json = None

# File containing the structured resume
auditor_result_file = "resume_llm_structured.json"

# Try loading the resume file
if Path(auditor_result_file).exists():
    try:
        with open(auditor_result_file, "r", encoding="utf-8") as f:
            auditor_resume_json = json.load(f)

        print("✅ Resume JSON loaded successfully")

    except Exception as e:
        print(f"⚠️ Error loading resume file: {e}")
        print("Using example JD only.")

else:
    print(f"ℹ️ File '{auditor_result_file}' not found.")
    print("Place your structured resume JSON in the project folder.")

# Safety check
if auditor_resume_json is None:
    print("⚠️ WARNING: Resume JSON not loaded.")

# Target company for job search
target_company = "Google"

# Debug info
print("\nDebug Info:")
print("Resume loaded:", auditor_resume_json is not None)
print("JD loaded:", auditor_jd_json is not None)

# Example of accessing resume data
if auditor_resume_json:
    skills = auditor_resume_json["sections"]["skills"]
    experience = auditor_resume_json["sections"]["experience"]

    print("\nTop Skills:", skills[:5])
    print("Number of jobs:", len(experience))

✅ Resume JSON loaded successfully

Debug Info:
Resume loaded: True
JD loaded: True

Top Skills: ['Machine Learning', 'Data Analysis', 'Database Administration', 'Python', 'R']
Number of jobs: 2


## 🔍 Step 4: Execute Opportunity Scan

The Scout agent will perform multi-source searches and match opportunities.

In [12]:
import json
import time

# Get variables from the appropriate scope
auditor_resume_json = locals().get('auditor_resume_json') or globals().get('auditor_resume_json')
auditor_jd_json = locals().get('auditor_jd_json') or globals().get('auditor_jd_json')

# Validate inputs
if not isinstance(auditor_resume_json, dict) or not auditor_resume_json:
    raise ValueError("auditor_resume_json is missing or empty. Run Step 3 first.")

if not isinstance(auditor_jd_json, dict) or not auditor_jd_json:
    raise ValueError("auditor_jd_json is missing or empty. Run Step 3 first.")

# Construct the discovery prompt
resume_json_str = json.dumps(auditor_resume_json, ensure_ascii=False)
jd_json_str = json.dumps(auditor_jd_json, ensure_ascii=False)

discovery_prompt = f"""I am initiating an "Opportunity Scan" for the following candidate.

**Target Company:** {target_company}

**Candidate Resume JSON (START):**
{resume_json_str}
**Candidate Resume JSON (END)**

**Original Job Description JSON (START):**
{jd_json_str}
**Original Job Description JSON (END)**

**Your Mission:**

1. **Analyze the Resume JSON**: Review the candidate's skills, experience, projects, and education from the JSON above.

2. **Identify Strong Matches**: Find the candidate's core competencies and strongest skills.

3. **Search for Opportunities**: Use your web search capability to find:
   - 2-3 open roles at {target_company} matching the profile
   - 2-3 similar roles at competitors (e.g., Microsoft, Amazon, Meta)
   - 2-3 roles at startups matching the candidate's top skills

4. **Match Analysis**: For each job found, explain how it matches the candidate's profile from the Resume JSON.

**Deliverable:**
Provide a Markdown table with columns:
- Job Title
- Company
- Match Level (High: >80%, Medium: 50-80%, Strategic Pivot)
- Why it Matches (reference specific skills/experience from JSON)
- Source Link (actual URL from your web search)

**CRITICAL:** Use web search to find real current job postings with actual URLs. Reference specific skills from the Resume JSON."""

print("🚀 Executing Opportunity Scan...")
print("=" * 60)
print(f"📋 Prompt includes:")
print(f"   - Resume JSON length: {len(resume_json_str)} characters")
print(f"   - JD JSON length: {len(jd_json_str)} characters")
print(f"   - Target Company: {target_company}")
print()

print("📡 Sending request to OpenRouter (Perplexity with web search)...")
print("⏳ This may take 30-60 seconds due to live web browsing...")

try:
    # Use OpenRouter with Perplexity model (has built-in web search)
    response = client.create_message(
        model=SCOUT_MODEL,
        messages=[
            {"role": "system", "content": SCOUT_SYSTEM_PROMPT},
            {"role": "user", "content": discovery_prompt}
        ],
        temperature=0.7,
        max_tokens=3000
    )
    
    print(f"✅ Response received!")

except Exception as e:
    print(f"❌ API Error: {e}")
    import traceback
    traceback.print_exc()


🚀 Executing Opportunity Scan...
📋 Prompt includes:
   - Resume JSON length: 3362 characters
   - JD JSON length: 305 characters
   - Target Company: Google

📡 Sending request to OpenRouter (Perplexity with web search)...
⏳ This may take 30-60 seconds due to live web browsing...
❌ Authentication Failed!
   Error: {'error': {'message': 'User not found.', 'code': 401}}

🔧 Fix:
   1. Get a valid key from: https://openrouter.ai/keys
   2. Make sure you have credits: https://openrouter.ai/credits
   3. Update your .env file with the new key
   4. Restart the kernel and try again
❌ API Error: 401 Client Error: Unauthorized for url: https://openrouter.ai/api/v1/chat/completions


🚀 Executing Opportunity Scan...
📋 Prompt includes:
   - Resume JSON length: 3362 characters
   - JD JSON length: 305 characters
   - Target Company: Google

📡 Sending request to OpenRouter (Perplexity with web search)...
⏳ This may take 30-60 seconds due to live web browsing...
❌ Authentication Failed!
   Error: {'error': {'message': 'User not found.', 'code': 401}}

🔧 Fix:
   1. Get a valid key from: https://openrouter.ai/keys
   2. Make sure you have credits: https://openrouter.ai/credits
   3. Update your .env file with the new key
   4. Restart the kernel and try again
❌ API Error: 401 Client Error: Unauthorized for url: https://openrouter.ai/api/v1/chat/completions


Traceback (most recent call last):
  File "C:\Users\HP\AppData\Local\Temp\ipykernel_12848\2720608804.py", line 67, in <module>
    response = client.create_message(
        model=SCOUT_MODEL,
    ...<5 lines>...
        max_tokens=3000
    )
  File "C:\Users\HP\AppData\Local\Temp\ipykernel_12848\1351448213.py", line 54, in create_message
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\HP\AppData\Local\Programs\Python\Python314\Lib\site-packages\requests\models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 401 Client Error: Unauthorized for url: https://openrouter.ai/api/v1/chat/completions


## 📊 Step 5: Retrieve and Display Results

In [6]:
import time
import json
from IPython.display import Markdown, display

# Display the response from Step 4
try:
    if 'response' not in locals():
        print("⚠️  No response available. Run Step 4 first to execute the opportunity scan.")
    else:
        print("📊 Scout Agent Response:")
        print("=" * 60)
        
        # Handle OpenRouter/OpenAI format response
        if isinstance(response, dict) and 'choices' in response:
            choice = response['choices'][0]
            message = choice.get('message', {})
            
            # Display the content
            content = message.get('content', '')
            if content:
                display(Markdown(content))
            else:
                print("Response received but no content found")
                print(f"Response: {json.dumps(response, indent=2, default=str)}")
                
            # Check for citations (Perplexity models include these)
            if 'citations' in response:
                print("\n📚 Sources:")
                for i, citation in enumerate(response['citations'][:5], 1):
                    print(f"   {i}. {citation}")
                    
        else:
            print("Warning: Unexpected response format")
            print(f"Response type: {type(response)}")
            print(str(response)[:500])
        
        print("=" * 60)
        
        # Save results to file
        output_file = f"scout_results_{int(time.time())}.json"
        result_data = {
            "target_company": target_company if 'target_company' in locals() or 'target_company' in globals() else None,
            "response": response,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
        }
        
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(result_data, f, indent=2, default=str)
        
        print(f"\n✅ Results saved to: {output_file}")
        
except Exception as e:
    print(f"❌ Error displaying response: {e}")
    import traceback
    traceback.print_exc()
    print("\n💡 Tip: Make sure you've run Step 4 first to get the response.")


⚠️  No response available. Run Step 4 first to execute the opportunity scan.
